# **MÓDULO 39 - Exercicio - Aplicando XGboost**

Nessa atividade vocês usarão uma base já connhecida, a base de propensão a compra de carros.
Já utilizamos essa base antes, nessa atividade, você deve carregar a base, realizar os tratamentos necessários, rodar o XGBoost, encontrar as principais variaveis.
Vamos lá!

In [ ]:
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_score, recall_score, f1_score, roc_auc_score)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from IPython.display import display


# 1. Carrega a base de dados, verifique os tipos de dados, dados faltantes e drop a coluna ID.


In [ ]:
# Carregamento da base
# Deixe o arquivo car_data.csv na mesma pasta do notebook.
df = pd.read_csv('car_data.csv')

print('Dimensões da base:', df.shape)
print('\nPrimeiras linhas:')
display(df.head())

print('\nTipos das colunas:')
print(df.dtypes)

print('\nDados faltantes por coluna:')
faltantes = df.isnull().sum()
print(faltantes)

# Caso existam valores ausentes, removemos as linhas incompletas.
# Na base original do exercício, a expectativa é não haver valores faltantes.
if faltantes.sum() > 0:
    df = df.dropna().reset_index(drop=True)
    print('\nLinhas com valores ausentes removidas.')

# Remoção da coluna de identificação, que não representa uma característica do cliente.
df = df.drop(columns=['User ID'])

print('\nBase após remover User ID:')
display(df.head())


# 2. Aplique o Label Encoder para a coluna Gender e exclua a coluna categórica.

In [ ]:
# Codificação da variável categórica Gender
label_encoder = LabelEncoder()
df['Gender_encoded'] = label_encoder.fit_transform(df['Gender'])

# Mostra o mapeamento criado pelo LabelEncoder
mapeamento_gender = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print('Mapeamento do Gender:', mapeamento_gender)

# Exclui a coluna categórica original e mantém apenas a versão numérica
df = df.drop(columns=['Gender'])
df = df[['Gender_encoded', 'Age', 'AnnualSalary', 'Purchased']]

display(df.head())
print('\nTipos após o encoding:')
print(df.dtypes)


# 3. Faça a matriz de correlação e analise as váriaveis que parecem ter maior correlação com a váriavel target.

In [ ]:
# Matriz de correlação
matriz_correlacao = df.corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(matriz_correlacao, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Matriz de Correlação')
plt.tight_layout()
plt.show()

# Correlação das variáveis com o target
correlacao_target = (matriz_correlacao['Purchased']
                     .drop('Purchased')
                     .sort_values(key=lambda s: s.abs(), ascending=False))

print('Correlação das variáveis com Purchased:')
print(correlacao_target)

print('\nVariáveis com maior correlação absoluta com o target:')
for variavel, valor in correlacao_target.items():
    print(f'- {variavel}: {valor:.4f}')


# 4. Faça a separação da base em X e Y e em seguida nas bases de treino e teste.

In [ ]:
# Separação entre variáveis independentes (X) e target (y)
X = df.drop(columns=['Purchased'])
y = df['Purchased']

# Divisão em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)
print('y_train:', y_train.shape)
print('y_test :', y_test.shape)
print('\nDistribuição do target no treino:')
print(y_train.value_counts(normalize=True).sort_index())


# 5. Treino o modelo Xgboost com sua base de treino.
Pesquise alguns hyperparametros na bibllioteca para utilizar, utilize ao menos 2.

In [ ]:
# Modelo XGBoost para classificação binária
# Hiperparâmetros configurados manualmente:
# - n_estimators: quantidade de rodadas/árvores de boosting
# - max_depth: profundidade máxima de cada árvore
# - learning_rate: tamanho do passo de atualização do boosting
modelo_xgb = xgb.XGBClassifier(
    objective='binary:logistic',
    n_estimators=100,
    max_depth=3,
    learning_rate=0.10,
    random_state=42,
    eval_metric='logloss'
)

modelo_xgb.fit(X_train, y_train)
print(modelo_xgb)


# 6. Realize as previsões para a base de teste e traga as probabilidades das previsões.

In [ ]:
# Classes previstas diretamente pelo modelo
classes_previstas = modelo_xgb.predict(X_test)

# Probabilidades para cada classe
probabilidades = modelo_xgb.predict_proba(X_test)
probabilidade_classe_1 = probabilidades[:, 1]

resultado_previsoes = pd.DataFrame({
    'Valor_real': y_test.to_numpy(),
    'Classe_prevista': classes_previstas,
    'Probabilidade_classe_0': probabilidades[:, 0],
    'Probabilidade_classe_1': probabilidade_classe_1
})

display(resultado_previsoes.head(10))


# 7. Faça a transformação das probabilidades nas previsões binárias e avalie o desempenho do modelo.

In [ ]:
# Conversão manual das probabilidades em classes binárias usando limiar de 0.5
limiar = 0.50
previsoes_binarias = (probabilidade_classe_1 >= limiar).astype(int)

# Métricas de avaliação
acuracia = accuracy_score(y_test, previsoes_binarias)
precisao = precision_score(y_test, previsoes_binarias)
recall = recall_score(y_test, previsoes_binarias)
f1 = f1_score(y_test, previsoes_binarias)
auc = roc_auc_score(y_test, probabilidade_classe_1)

print(f'Acurácia : {acuracia:.4f}')
print(f'Precisão : {precisao:.4f}')
print(f'Recall   : {recall:.4f}')
print(f'F1-score : {f1:.4f}')
print(f'ROC AUC  : {auc:.4f}')

print('\nRelatório de classificação:')
print(classification_report(y_test, previsoes_binarias))

# Matriz de confusão
cm = confusion_matrix(y_test, previsoes_binarias)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Matriz de Confusão')
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.tight_layout()
plt.show()


# 8. Traga a lista com a feature importance.
As features mais importantes são as mesmas que indicaram maior correlação na matriz de correlação?

In [ ]:
# Feature importance do XGBoost
feature_importance = (pd.Series(modelo_xgb.feature_importances_, index=X.columns)
                      .sort_values(ascending=False))

print('Feature importance:')
print(feature_importance)

plt.figure(figsize=(7, 4))
feature_importance.sort_values().plot(kind='barh')
plt.title('Feature Importance - XGBoost')
plt.xlabel('Importância')
plt.ylabel('Variável')
plt.tight_layout()
plt.show()

# Comparação com a correlação calculada na etapa 3
ordem_correlacao = correlacao_target.abs().sort_values(ascending=False).index.tolist()
ordem_importancia = feature_importance.index.tolist()

print('Ordem por correlação absoluta com Purchased:')
print(ordem_correlacao)
print('\nOrdem por feature importance do XGBoost:')
print(ordem_importancia)

coincidem_top2 = set(ordem_correlacao[:2]) == set(ordem_importancia[:2])
print('\nAs duas variáveis de maior destaque coincidem?', coincidem_top2)


### Análise final

A matriz de correlação permite observar a associação **linear** de cada variável com o target `Purchased`. Já o `feature_importance_` do XGBoost mede a contribuição das variáveis para as divisões realizadas pelas árvores do modelo, capturando também relações não lineares e interações entre atributos.

Nesta base, a comparação deve mostrar que **Age** e **AnnualSalary** concentram a maior parte da informação preditiva, enquanto **Gender_encoded** tende a ter participação bem menor. Mesmo que a ordem exata entre `Age` e `AnnualSalary` possa variar entre correlação e importância, isso não representa contradição: correlação e feature importance medem aspectos diferentes da relação entre as variáveis e o alvo.

Assim, o resultado do XGBoost é coerente com a análise exploratória quando as variáveis que mais se relacionam com a compra também aparecem entre as mais importantes para o modelo.
